# Prepare full word-trial EEG once: ZuCo + TeCo
**CPU → Runtime → Run all.** This notebook prepares data only; it does not train classifiers.

It saves each usable first-stored-fixation epoch with **all channels, 125-Hz time courses (8-ms samples)**, and exact word/event metadata. New preparation uses average reference, whole-sentence 40-Hz low-pass and polyphase anti-alias resampling to 125 Hz, −200…800 ms, baseline −200…0 excluding zero. Original 30-Hz results remain separate.

All sentiment labels, neutral words, function words, disagreements and control **reading** remain eligible for signal preservation. High-amplitude and short-duration trials are flagged rather than silently discarded. Missing/invalid fixations and impossible epoch boundaries have explicit metadata outcomes.

**Storage:** target about 8–9 GB, capped at 9 decimal GB with 0.5 GB free-space reserve. The notebook computes a conservative bound from saved onset tables and verifies Google Drive quota before source extraction. No raw-source copy is added to Drive. Allow mounting and authenticate with the **same Google account** for the quota check. No GitHub token or GPU is needed.

Output: `Thesis/Results/word_sentiment_eeg/trial_archive_125hz/`. Rerunning verifies committed chunks and resumes; it does not append duplicate trials. Later classifiers can choose labels and features from this archive.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json
from IPython.display import display, Markdown
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
REPO=Path('/content/word-sentiment-eeg-trial-preparation')
RUN=THESIS/'Results/word_sentiment_eeg/trial_archive_125hz'
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout has edits. Preserve them before updating.')
    git('-C',REPO,'pull','--ff-only','origin','main')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-colab.txt')],check=True)
print('Code:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
BASE=[sys.executable,'-u',str(REPO/'prepare_trial_archive.py'),'--thesis-root',str(THESIS)]
subprocess.run(BASE+['--plan-only'],check=True)
PLAN=json.loads((RUN/'preflight.json').read_text())
print(f"Uncompressed upper bound from saved availability: {PLAN['upper_bound_uncompressed_bytes']/1024**3:.1f} GiB")
print('Compression can reduce this; the capacity check does not assume it.')
display(pd.DataFrame(PLAN['records'])[['dataset','task','subject','word_rows','candidate_epoch_upper_bound']])


## Verify the mounted account and available storage
This checks the account that will own new files, using a tiny temporary probe in the output folder. It makes no changes to sharing or old data. Choose the same account you mounted above. If quota or account verification fails, extraction stops before reading source EEG.


In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
import time, uuid

auth.authenticate_user(clear_output=False)
creds,_=google.auth.default()
service=build('drive','v3',credentials=creds,cache_discovery=False)
about=service.about().get(fields='user(emailAddress),storageQuota(limit,usage)').execute()
email=about.get('user',{}).get('emailAddress')
if not email:
    raise RuntimeError('Cannot identify the quota account. No extraction started.')
probe_name='archive_quota_probe_'+uuid.uuid4().hex+'.txt'
probe=RUN/probe_name
probe.write_text('Temporary account verification for trial archive.')
verified=False
try:
    for attempt in range(12):
        found=service.files().list(q="name='"+probe_name+"' and trashed=false",
                                  fields='files(id,owners(emailAddress,me))',pageSize=10).execute().get('files',[])
        if found:
            owners=found[0].get('owners',[])
            if not any(x.get('me') and x.get('emailAddress','').lower()==email.lower() for x in owners):
                raise RuntimeError('Quota account differs from mounted-file owner. Reauthenticate with the mounted account.')
            verified=True
            break
        time.sleep(3)
finally:
    probe.unlink(missing_ok=True)
if not verified:
    raise RuntimeError('Mounted-file owner was not verified (account mismatch or pending Drive upload). No extraction started; rerun this cell after checking accounts.')
quota=about['storageQuota']
if 'limit' not in quota:
    FREE_DRIVE_BYTES=2**63-1
    print('Verified account:',email,'; Drive reports no individual quota limit.')
else:
    FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage','0')))
    print('Verified account:',email,'; free:',round(FREE_DRIVE_BYTES/1024**3,2),'GiB')


## Prepare the full trial archive
This can take hours. Every trial and channel remains separate; only the sample rate is reduced. A 9-GB archive cap prevents exceeding the storage budget. Saved onsets are reused only when source hashes and full word segments agree. Onsets not searched previously (for example TeCo controls) are recovered now. No later fixation replaces a missing first fixation.

Keep one extraction notebook active. Every committed block has hashes and explicit event-to-array indices. A stopped run remains incomplete; rerun to resume it. Progress messages distinguish inspected sentences, committed blocks and completed reader recordings.


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=BASE+['--allow-cloud-download','--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,RUN/'runtime_status.json',activity='Full individual-trial EEG preparation')


## Completion and what was saved
Only `manifest.json: status=completed` means all 48 expected reader/task recordings finished. NPZ shards contain `eeg_uV[trial,channel,time]`, `baseline_uV[trial,channel]`, `event_id`, `times_ms`, `channel_names`. Each metadata row points to one shard and one row, or states why no epoch exists.

Revised TeCo three-level labels are attached if already on Drive. Otherwise historical labels are preserved and missing grammar/revised labels are explicitly unknown. They can be joined later by sentence/word identity without reprocessing EEG.

This preserves full epochs for future feature choices. Continuous-data deconvolution, different epoch boundaries, later-fixation analyses, or frequencies removed by 40-Hz/upstream filters can still need original source files.


In [ ]:
manifest=json.loads((RUN/'manifest.json').read_text())
print('Status:',manifest['status'])
print('Completed:',manifest['completed_reader_recordings'],'/',manifest['expected_reader_recordings'])
if manifest['status']!='completed':
    raise RuntimeError('Preparation is incomplete. Inspect manifest failures; saved blocks are retained for resume.')
display(pd.read_csv(RUN/'reader_coverage.csv'))
display(Markdown((RUN/'README.md').read_text()))
print('Archive folder:',RUN)
